# KAT-VAD v2 — **Nexar N0**: mirror + census (no features, no scores)

Plan `.project/plans/katvad-v2-nexar-feasibility.md` N0. Dataset `nexar-ai/nexar_collision_prediction` (gated, Nexar
Open Data License: attribution, no redistribution, no re-identification). **CPU runtime is enough** — nothing here
encodes a frame.

| step | what | where |
|---|---|---|
| 0 | setup, deps, HF login from the Colab secret `HF_TOKEN` | CPU |
| 1 | mirror the **whole** HF repo (train 1,500 mp4 + test-public/private + csv) → `Thesis/data/Nexar/raw/`, size-check every file | network |
| 2 | `core.tools.nexar_census`: metadata.csv × container headers (PyAV, no decode) + 20 full decodes → `census.{json,md}` | CPU |
| 3 | dry-run of the split freeze (prints counts only; the real freeze runs locally from `census.json` and is committed) | CPU |

**Why mirror to Drive, not stream:** DoTA's pixels vanished with their links. Keeping the raw mp4 on Drive (~31 GB)
costs less than losing them. Step 1 is resumable: files already on Drive with the HF size are skipped.

**Nothing is decided here.** The census is metadata; the split rule is in `core/tools/nexar_splits.py`'s docstring and
was written before this notebook ran. **Paste `census.md` to Claude** after step 2.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # data/, cache/
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # kat-vad/ (code), outputs/ (v2 read-outs)
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'NEXAR_HF_REPO'), 'this upload predates Nexar N0 -- re-upload branch v2'

RAW = constants.DATA_ROOT / constants.NEXAR_DATASET / 'raw'
OUT = constants.OUTPUT_ROOT / 'v2' / 'REPORTS' / 'nexar_n0'
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print('raw  ->', RAW)
print('out  ->', OUT)

In [ ]:
%%bash
pip install -q "huggingface_hub>=0.25" hf_transfer av
python -c "import av, huggingface_hub, hf_transfer; print('av', av.__version__, 'hub', huggingface_hub.__version__)"
df -h /content | tail -1

In [ ]:
import json
import subprocess
import time

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))   # the secret must be enabled for this notebook
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
ENV = dict(os.environ)


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')

## 1. Mirror HF → Drive, size-check every file

If this fails with 401/403, open the dataset page while logged in, accept the license, and check that the `HF_TOKEN`
secret is a read token of that account.

In [ ]:
from huggingface_hub import HfApi, snapshot_download

api = HfApi()
remote = {f.path: f.size for f in api.list_repo_tree(constants.NEXAR_HF_REPO, repo_type='dataset',
                                                     recursive=True, expand=True)
          if getattr(f, 'size', None) is not None}
n_mp4 = {d: sum(p.startswith(d) and p.endswith('.mp4') for p in remote) for d in ('train/', 'test-public/', 'test-private/')}
print(f'{len(remote)} files, {sum(remote.values()) / 1e9:.1f} GB; mp4 per folder {n_mp4}')
assert n_mp4['train/'] == constants.NEXAR_VIDEOS, n_mp4

t0 = time.time()
snapshot_download(constants.NEXAR_HF_REPO, repo_type='dataset', local_dir=str(RAW), max_workers=8)
print(f'download step {time.time() - t0:.0f}s')

bad = {p: (s, (RAW / p).stat().st_size if (RAW / p).exists() else None)
       for p, s in remote.items() if not (RAW / p).exists() or (RAW / p).stat().st_size != s}
assert not bad, f'{len(bad)} files missing or wrong size: {list(bad.items())[:5]}'
(RAW / 'nexar_files_manifest.json').write_text(json.dumps(remote, indent=1, sort_keys=True))
print(f'all {len(remote)} files match HF sizes')

## 2. Census — metadata × container headers (no decode except the 20-video check)

`--workers 16`: header probes are I/O-bound on Drive FUSE. Re-running overwrites `census.{json,md}` (atomic writes).

In [ ]:
run([sys.executable, '-m', 'core.tools.nexar_census', '--root', RAW, '--out-dir', OUT,
     '--workers', 16, '--verify-decode', 20], log=OUT / 'census.log')
print((OUT / 'census.md').read_text())

## 3. Dry-run of the split freeze (counts only)

Writes to `/content/nexar_splits_dryrun`, **not** to the repo. The committed freeze runs locally from the same
`census.json` (`python -m core.tools.nexar_splits --census outputs/v2/REPORTS/nexar_n0/census.json`) — same code, same
seed, same ids. If the census reports `probe_failed` or a count ≠ 1,500, **stop and paste to Claude** instead.

In [ ]:
DRY = Path('/content/nexar_splits_dryrun')
if DRY.exists():
    import shutil
    shutil.rmtree(DRY)
DRY.mkdir()
run([sys.executable, '-m', 'core.tools.nexar_splits', '--census', OUT / 'census.json', '--split-dir', DRY])
m = json.loads((DRY / constants.V2_NEXAR_MANIFEST_FILENAME).read_text())
print(json.dumps({k: m[k] for k in ('splits', 'per_label', 'with_issues', 'median_duration_s', 'position_edges')}, indent=1))

## 4. Record

Paste `census.md` (and the dry-run block above) to Claude. Sync `Thesis-V2/outputs/v2/REPORTS/nexar_n0/` to the local
repo's `outputs/v2/REPORTS/nexar_n0/` (it is tracked) — the split is frozen and committed from that `census.json`.